# Setup

In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

In [2]:
import os
import sys
from pprint import pprint

# so that mllm_shap can be imported without installing the package
sys.path.insert(0, os.path.abspath("../mllm_shap/src"))

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "0"
os.environ["LOG_LEVEL"] = "WARNING"

In [3]:
import numpy as np
import pandas as pd
import torch

np.random.seed(42)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Using device: {device}")

/Users/pawel.pozorski/Desktop/MLLM-Shap/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Using device: cpu


In [4]:
from mllm_shap.connectors import TransformersCausalText, ModelConfig
from mllm_shap.connectors.enums import ModelHistoryTrackingMode, Role, SystemRolesSetup
from mllm_shap.connectors.filters import ExcludePunctuationTokensFilter
from mllm_shap.shap import Explainer, PreciseShapExplainer
from mllm_shap.shap.embeddings import CustomEmbedding
from mllm_shap.shap.enums import Mode
from mllm_shap.shap.normalizers import AbsSumNormalizer
from mllm_shap.shap.similarity import TfIdfCosineSimilarity
from mllm_shap.utils.jupyter import display_shap_colors_df
from mllm_shap.shap.embeddings import (
    MeanReducer,
)

# Usage

Define a `TransformersCausalText` model (this call loads it into memory!).

Create a `PreciseShapExplainer` backed by an external `CustomEmbedding` model (`intfloat/e5-small-v2`). Embeddings are reduced via mean pooling, compared with TF-IDF weighted cosine similarity, and normalized with `AbsSumNormalizer` so absolute values sum to 1.

For multi-turn chats (second turn onward), `PreciseShapExplainer` is replaced with `McShapExplainer` (`num_samples=100`) to avoid exponential mask space growth.

In [5]:
model = TransformersCausalText(
    device=device, history_tracking_mode=ModelHistoryTrackingMode.TEXT
)  # track and generate only text history

In [6]:
external_emb = CustomEmbedding(
    generation_tokenizer=model.processor,
    embed_model_id="intfloat/e5-small-v2",
    embed_revision="ffb93f3bd4047442299a41ebb6fa998a38507c52",
    device=device,
    max_length=64,
    batch_size=64,
    l2_normalize=True,
    local_files_only=False,
)

In [7]:
shap = PreciseShapExplainer(
    mode=Mode.CONTEXTUAL,
    embedding_model=external_emb,
    embedding_reducer=MeanReducer(),
    similarity_measure=TfIdfCosineSimilarity(),  # use TF-IDF weighted cosine similarity to compare embeddings
    normalizer=AbsSumNormalizer(),  # use power-shift normalization with power of 2.0
)
explainer = Explainer(model=model, shap_explainer=shap)

Create a new chat with `SystemRolesSetup.NONE` — no turns are treated as system context, so all user tokens contribute to SHAP value calculation. `ExcludePunctuationTokensFilter` marks punctuation tokens (like `?`) as non-explainable — they are always present in every masked query and reduce the total number of mask evaluations needed.

In [8]:
df = pd.read_parquet("hf://datasets/Pawlo77/mllm-shap/single_sentence.parquet")
sample_entry = df.sample(30, random_state=42).iloc[25].to_dict()

sample_entry["sentences"][0]

'Who invented fantasy sports?'

In [9]:
chat = model.get_new_chat(
    system_roles_setup=SystemRolesSetup.NONE,
    token_filter=ExcludePunctuationTokensFilter(),  # exclude punctuation tokens from shapley values calculation
)

chat.new_turn(Role.USER)
chat.add_text(sample_entry["sentences"][0])
chat.end_turn()
chat.refresh(full=True)

Let's have a look at chat representation:

In [10]:
chat.get_conversation()

[[ChatEntry(content_type=0, roles=[USER, USER, USER, USER, USER], content='Who,  invented,  fantasy,  sports, ?', shap_values=None)]]

Representation is a list of list of ConversationEntry - so it can be accessed as  chat.get_conversation()[turn_number][message_number].{field}

Calculate Shapley Values for the current conversation.

`verbose=True` grants access to the `history` object (described below). `generation_kwargs` limits generation to 16 new tokens with `text_temperature=0.0` for fully deterministic output.

In [11]:
generation_kwargs = {
    "max_new_tokens": 16,
    "model_config": ModelConfig(
        text_temperature=0.0,
        audio_temperature=None,
        audio_top_k=None,
    ),
}

result = explainer(
    chat=chat,
    verbose=True,
    generation_kwargs=generation_kwargs,
    progress_bar=True,
)

Precise SHAP:   0%|          | 0/15 [00:00<?, ?it/s]

`result` has the following fields:

- `full_chat` — chat with the base response (generated from the full, unmasked input) and with SHAP values attached
- `source_chat` — the original chat fed to the explainer
- `history` — list of all (mask, mask_hash, masked_chat, model_response) tuples evaluated during explanation

The cache stored in `full_chat` holds SHAP values, embeddings, and masks. On subsequent calls it is reused to skip already-evaluated masks.

`history` has 14 entries: `ExcludePunctuationTokensFilter` leaves 4 explainable tokens (`Who`, `invented`, `fantasy`, `sports`), so the Precise explainer enumerates all 2⁴ − 2 = 14 non-trivial subsets. Each entry is a tuple of:

- mask (1D bool tensor)
- mask hash
- masked chat (`None` if response came from cache)
- model response

Let's see all masked prompts evaluated:

In [12]:
[c[2].decode_text() if c is not None else None for c in result.history]

[' sports?',
 ' fantasy?',
 ' fantasy sports?',
 ' invented?',
 ' invented sports?',
 ' invented fantasy?',
 ' invented fantasy sports?',
 'Who?',
 'Who sports?',
 'Who fantasy?',
 'Who fantasy sports?',
 'Who invented?',
 'Who invented sports?',
 'Who invented fantasy?']

`?` is never masked out because `ExcludePunctuationTokensFilter` marks it as non-explainable — it is always kept in every masked query. With `SystemRolesSetup.NONE` there are no system tokens, so only user tokens are present and only the non-punctuation ones are masked.

Let's now analyze the calculated Shapley Values.

In [13]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[USER, USER, USER, USER, USER], content='Who,  invented,  fantasy,  sports, ?', shap_values=[0.13283731043338776, 0.31852975487709045, 0.3015713095664978, 0.24706169962882996, nan])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='\n, The,  first,  fantasy,  sports,  league,  was,  created,  in,  the,  early,  1900, s,  by,  a,  ...', shap_values=[nan, nan, ..., nan, nan])]]


The model tracks only text tokens. `shap_values` is now populated: `?` has `NaN` because it was excluded by `ExcludePunctuationTokensFilter`. The 4 non-NaN tokens are `Who`, `invented`, `fantasy`, `sports`.

In [14]:
user_entry = explained_chat_conversation[0][0]

display_shap_colors_df(
    pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
)

,Token,Shapley Value,Role
0,Who,0.132837,0
1,invented,0.318530,0
2,fantasy,0.301571,0
3,sports,0.247062,0
4,?,nan,0


Let's create another turn to see how input significance will change:

In [15]:
explained_chat.new_turn(Role.USER)
explained_chat.add_text("Can you repeat?")
explained_chat.end_turn()

Adding a second user turn increases the total explainable token count to ~23. `PreciseShapExplainer` would require up to 2²³ ≈ 8 M evaluations — infeasible. We switch to `McShapExplainer` with `num_samples=100` for an approximate result. The cache from the first explanation is cleared first since it was calculated by a different explainer instance.

In [16]:
# For multi-turn chats, use Monte Carlo sampling instead of Precise enumeration
# to avoid exponential mask space explosion (2^23 masks would be infeasible)
from mllm_shap.shap import McShapExplainer, Explainer
from mllm_shap.shap.enums import Mode
from mllm_shap.shap.embeddings import MeanReducer
from mllm_shap.shap.similarity import TfIdfCosineSimilarity
from mllm_shap.shap.normalizers import AbsSumNormalizer

# Clear the cache from the Precise explainer before using a different explainer
explained_chat.cache = None

mc_shap = McShapExplainer(
    num_samples=100,  # Monte Carlo: sample 100 masks instead of all 2^23
    mode=Mode.CONTEXTUAL,
    embedding_model=external_emb,
    embedding_reducer=MeanReducer(),
    similarity_measure=TfIdfCosineSimilarity(),
    normalizer=AbsSumNormalizer(),
)

mc_explainer = Explainer(model=model, shap_explainer=mc_shap)

result = mc_explainer(
    chat=explained_chat,
    verbose=True,
    generation_kwargs=generation_kwargs,
    progress_bar=True,
)

Monte Carlo SHAP:   0%|          | 0/100 [00:00<?, ?it/s]

In [17]:
[c[2].decode_text() if c is not None else None for c in result.history]

[' invented fantasy sports?\nThe first fantasy sports league was created in the early 1900s by a groupCan you repeat?',
 'Who fantasy sports?\nThe first fantasy sports league was created in the early 1900s by a groupCan you repeat?',
 'Who invented sports?\nThe first fantasy sports league was created in the early 1900s by a groupCan you repeat?',
 'Who invented fantasy?\nThe first fantasy sports league was created in the early 1900s by a groupCan you repeat?',
 'Who invented fantasy sports?The first fantasy sports league was created in the early 1900s by a groupCan you repeat?',
 'Who invented fantasy sports?\n first fantasy sports league was created in the early 1900s by a groupCan you repeat?',
 'Who invented fantasy sports?\nThe fantasy sports league was created in the early 1900s by a groupCan you repeat?',
 'Who invented fantasy sports?\nThe first sports league was created in the early 1900s by a groupCan you repeat?',
 'Who invented fantasy sports?\nThe first fantasy league was c

In [18]:
explained_chat = result.full_chat

explained_chat_conversation = explained_chat.get_conversation()
pprint(explained_chat_conversation)

[[ChatEntry(content_type=0, roles=[USER, USER, USER, USER, USER], content='Who,  invented,  fantasy,  sports, ?', shap_values=[0.020872026681900024, 0.01622156985104084, 0.04309394210577011, 0.03333008661866188, nan])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='\n, The,  first,  fantasy,  sports,  league,  was,  created,  in,  the,  early,  1900, s,  by,  a,  ...', shap_values=[0.019314663484692574, 0.0029069234151393175, ..., 0.03757157176733017, 0.03486331179738045])],
 [ChatEntry(content_type=0, roles=[USER, USER, USER, USER], content='Can,  you,  repeat, ?', shap_values=[0.07163805514574051, 0.05544786527752876, 0.038045484572649, nan])],
 [ChatEntry(content_type=0, roles=[ASSISTANT, ASSISTANT, ..., ASSISTANT, ASSISTANT], content='\n, \n, Answer, :,  The,  first,  fantasy,  sports,  league,  was,  created,  in,  the,  early,  190...', shap_values=[nan, nan, ..., nan, nan])]]


In [19]:
dt = []
for i in (0, 2):
    user_entry = explained_chat_conversation[i][0]
    df = pd.DataFrame(
        list(zip(user_entry.content, user_entry.shap_values, user_entry.roles)),
        columns=["Token", "Shapley Value", "Role"],
    )
    df["Turn"] = i
    dt.append(df)

df = pd.concat(dt).reset_index(drop=True)
display_shap_colors_df(df)

,Token,Shapley Value,Role,Turn
0,Who,0.020872,0,0
1,invented,0.016222,0,0
2,fantasy,0.043094,0,0
3,sports,0.033330,0,0
4,?,nan,0,0
5,Can,0.071638,0,2
6,you,0.055448,0,2
7,repeat,0.038045,0,2
8,?,nan,0,2
